<a href="https://colab.research.google.com/github/pedroabn/portfolio/blob/Geral/Python/leituraPDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Baixar pacote externo

In [ ]:
!pip install pdfplumber

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.0 MB/s eta 0:00:00


Baixa os pacotes para realizar as atividades

In [ ]:
import pdfplumber as pp
import re
from google.colab import drive,files
from os.path import split
import os
import pandas as pd
import numpy as np

Acesso ao drive do Google

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


Retornar os PDFs vistos dentro da pasta do Drive

In [ ]:
id_pdf = ["/content/drive/MyDrive/ltdoc/1.pdf",
          "/content/drive/MyDrive/ltdoc/2.pdf","/content/drive/MyDrive/ltdoc/3.pdf",
          "/content/drive/MyDrive/ltdoc/4.pdf","/content/drive/MyDrive/ltdoc/5.pdf",
          "/content/drive/MyDrive/ltdoc/6.pdf","/content/drive/MyDrive/ltdoc/7.pdf",
          "/content/drive/MyDrive/ltdoc/8.pdf","/content/drive/MyDrive/ltdoc/9.pdf",
          "/content/drive/MyDrive/ltdoc/10.pdf","/content/drive/MyDrive/ltdoc/11.pdf",
          "/content/drive/MyDrive/ltdoc/12.pdf","/content/drive/MyDrive/ltdoc/13.pdf",
          "/content/drive/MyDrive/ltdoc/14.pdf","/content/drive/MyDrive/ltdoc/15.pdf",
          "/content/drive/MyDrive/ltdoc/16.pdf","/content/drive/MyDrive/ltdoc/18.pdf",
          "/content/drive/MyDrive/ltdoc/19.pdf","/content/drive/MyDrive/ltdoc/20.pdf",
          "/content/drive/MyDrive/ltdoc/21.pdf","/content/drive/MyDrive/ltdoc/22.pdf",
          "/content/drive/MyDrive/ltdoc/23.pdf","/content/drive/MyDrive/ltdoc/24.pdf",
          "/content/drive/MyDrive/ltdoc/25.pdf","/content/drive/MyDrive/ltdoc/26.pdf",
          "/content/drive/MyDrive/ltdoc/27.pdf","/content/drive/MyDrive/ltdoc/28.pdf",
          "/content/drive/MyDrive/ltdoc/17.pdf"]

Leitura do nome do projeto

In [ ]:
def ler_nome(caminho_pdf):
#Usa a lib para abrir o pdf, o qual possui o caminho definido na célula de cima
# e lê as primeiras páginas em busca do objeto do projeto
  with pp.open(caminho_pdf) as pdf:
        page = pdf.pages[0]
        text = page.extract_text()
        linhas = text.split('\n')
  for linha in linhas:
    if re.search(r'\bNome\b', linha, re.IGNORECASE):
      nome = re.search(r'Nome do Projeto\s*[:\-]?\s*(.+)', text, re.IGNORECASE)
      if nome:
        return nome.group(1).strip()

  else:
        print("❌ Padrão não encontrado.")
        return None

Teste da def

In [ ]:
j = "/content/drive/MyDrive/ltdoc/17.pdf"
print(ler_nome(j))

Ação, Sopapo & Traição


Leitura do resumo do projeto

In [ ]:
def ler_objetivo(caminho_pdf):
#Usa a lib para abrir o pdf, o qual possui o caminho definido na célula de cima
# e lê as 3 primeiras páginas em busca do objeto do projeto
  with pp.open(caminho_pdf) as pdf:
        page = pdf.pages[0-5]
        text = ""
        for i in range(5):
            if i < len(pdf.pages):
                page = pdf.pages[i]
                text += page.extract_text()
        linhas = text.split('\n')
#Lê linha por linha do Objeto do projeto até objetivo geral
  for linha in linhas:
    obj = re.search(
        r'Objeto do Projeto[:\-]?\s*(.*?)\s*Objetivo Geral\s*[:\-]?',
        text,re.DOTALL)
    if obj:
        objetivo = obj.group(1).strip()
        return objetivo
  else:
        print("❌ Padrão não encontrado.")
        return None

Teste da def

In [ ]:
j = "/content/drive/MyDrive/ltdoc/13.pdf"
# print(ler_objetivo(j))

Leitura de subdivisão do audiovisual

In [ ]:
def ler_subdivisao(objetivo):
    #Criação de um dicionário para retornar os valores padronizados
  termos = {
        "Curta metragem": ["videoclipes","curta metragem", "curta-metragem"],
        "Média metragem": ["média metragem", "média-metragem","media metragem"],
        "Longa metragem": ["longa metragem", "longa-metragem","série","obra seriada"],
        "Oficina": ["oficina","formação","ensinar"],
        "Programações e ações": ["evento","concurso"],
        "Difusão": ["festival","mostra de cinema","sessões"],
        "Documentário": ["documentário","documental"],
        "Desenvolvimento": ["pós-produção","pré-produção","composição",
                            "desenvolver roteiro","desenvolver projeto"],
        "Animação": ["animação","desenhho"],
        "Memória e Preservação":["acervo","catalogação","catalogar","conservar"]}
    #Se há leitura de um objetivo geral, ele irá retornar tudo em minúsculo para
    #evitar qualquer dificuldade de leitura
  if objetivo:
    ol = objetivo.lower()
  else:
    return ""
    #Criar um dicionário para identificar posição e as categorias vistas,
    #retornando o primeiro termo. A escolha do primeiro termo, é devido a repeti
    #ção dos termos e buscando objetividade da escolha.
  posicoes = {}
  for categoria, variacoes in termos.items():
        for var in variacoes:
            pos = ol.find(var)
            if pos != -1:
                if categoria not in posicoes or pos < posicoes[categoria]:
                    posicoes[categoria] = pos
  if posicoes:
        if posicoes:
          ordenados = sorted(posicoes.items(), key=lambda x: x[1])  # ordena por posição
          dp = [item[0] for item in ordenados[:2]]
          if len(dp) == 2:
            return dp[0], None
          elif len(dp) == 1:
            return dp[0], None
          else:
            return None, None
  else:
        return None

Teste da def

In [ ]:
j = "/content/drive/MyDrive/ltdoc/7.pdf"
print(ler_subdivisao(ler_objetivo(j)))

('Documentário', None)


Criação de uma tabela, com retorno das defs em colunas para analisar e fazer o join com a planilha do SIC

In [ ]:
dados = []
for caminho_pdf in id_pdf:
  try:
      print(f"Lendo arquivo: {caminho_pdf}")
      aqvname = os.path.basename(caminho_pdf)
      pdfnome = ler_nome(caminho_pdf)
      print(f"Nome extraído: {pdfnome}")
      pdfobjetivo = ler_objetivo(caminho_pdf)
      objetivo = pdfobjetivo
      pdfsubdivisao = ler_subdivisao(objetivo)
      print(f"Subdivisão extraída: {pdfsubdivisao}")
      dados.append({
        "arquivo": aqvname,
        "nome": pdfnome if pdfnome else "",
        "objetivo": pdfobjetivo if pdfobjetivo else "",
        "subdivisao":pdfsubdivisao })
  except Exception as e:
      print(f"Erro ao processar o arquivo {caminho_pdf}: {e}")

Lendo arquivo: /content/drive/MyDrive/ltdoc/1.pdf
Nome extraído: FICCIONALIZAR NA MATA
Subdivisão extraída: ('Oficina', None)
Lendo arquivo: /content/drive/MyDrive/ltdoc/2.pdf
Nome extraído: OS PIONEIROS DA AVIAÇÃO
Subdivisão extraída: ('Longa metragem', None)
Lendo arquivo: /content/drive/MyDrive/ltdoc/3.pdf
Nome extraído: OS GUERREIROS DA RUA – O FILME
Subdivisão extraída: ('Desenvolvimento', None)
Lendo arquivo: /content/drive/MyDrive/ltdoc/4.pdf
Nome extraído: Herdeiros do Silêncio: Capital, Tráfico e Poder no Recife Oitocentista
Subdivisão extraída: ('Documentário', None)
Lendo arquivo: /content/drive/MyDrive/ltdoc/5.pdf
Nome extraído: Distribuição do longa-metragem Armorialma
Subdivisão extraída: ('Longa metragem', None)
Lendo arquivo: /content/drive/MyDrive/ltdoc/6.pdf
Nome extraído: DENTRO DA ESCURIDÃO
Subdivisão extraída: ('Desenvolvimento', None)
Lendo arquivo: /content/drive/MyDrive/ltdoc/7.pdf
Nome extraído: De Dunquerque a Olinda: a Loucura do Carnaval” (doc)
Subdivisão ex

In [ ]:
df = pd.DataFrame(dados)
print(f"\n Arquivo processado com sucesso.")


 Arquivo processado com sucesso.


In [ ]:
if dados:
    df.to_excel("Subdivisao.xlsx", index=False)
    print("✅ Subdivisao.xlsx salvo.")
else:
    print("⚠️ Nenhum dado encontrado.")

files.download("Subdivisao.xlsx")

✅ Subdivisao.xlsx salvo.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>